In [ ]:
# Settings: use GPU T4 x2, and add the prepare notebook's output as an input.
# Each language in LANGUAGES trains on its own GPU.
LANGUAGES = ["English", "Telugu"]
MAX_HOURS = 10
RESTORE_STEP = {"English": 0, "Telugu": 0}

In [ ]:
import glob, os, shutil, subprocess, time
REPO = "https://github.com/23f2001033/IITH-Intern-Assignment-.git"
PROJECT = "/tmp/project"
shutil.rmtree(PROJECT, ignore_errors=True)
subprocess.run(["git", "clone", "--depth", "1", REPO, PROJECT], check=True)
print(subprocess.run(["git", "-C", PROJECT, "log", "-1", "--format=%h %s"], capture_output=True, text=True).stdout)
subprocess.run("pip install -q pyworld tgt g2p-en inflect unidecode", shell=True, check=True)

In [ ]:
# Unpack the features. Checkpoints and logs live in /kaggle/working so Kaggle keeps them.
subprocess.run("nvidia-smi", shell=True)
for language in LANGUAGES:
    tar = glob.glob("/kaggle/input/**/{}_preprocessed.tar".format(language), recursive=True)[0]
    work = "/tmp/tts/" + language
    os.makedirs(work, exist_ok=True)
    subprocess.run(["tar", "-xf", tar, "-C", work], check=True)
    for name in ("ckpt", "log"):
        keep = "/kaggle/working/{}/{}".format(language, name)
        os.makedirs(keep, exist_ok=True)
        if not os.path.exists(work + "/" + name):
            os.symlink(keep, work + "/" + name)
    print(language, open(work + "/preprocessed_data/feature_summary.json").read())

In [ ]:
# Train both models at once, one per GPU
os.makedirs("/kaggle/working/logs", exist_ok=True)
jobs = {}
for gpu, language in enumerate(LANGUAGES):
    command = ["python", "{}/{}/train.py".format(PROJECT, language), "--work_dir", "/tmp/tts/" + language,
               "--max_hours", str(MAX_HOURS), "--restore_step", str(RESTORE_STEP[language])]
    log = open("/kaggle/working/logs/train_{}.log".format(language), "a")
    jobs[language] = subprocess.Popen(command, stdout=log, stderr=subprocess.STDOUT,
                                      env=dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu)))

def last_line(language):
    path = "/kaggle/working/{}/log/train/log.txt".format(language)
    return open(path).readlines()[-1].strip() if os.path.exists(path) else "(no steps logged yet)"

while any(p.poll() is None for p in jobs.values()):
    time.sleep(600)
    for language in jobs:
        print(time.strftime("%H:%M"), language, last_line(language), flush=True)
print({k: p.returncode for k, p in jobs.items()})
for language in jobs:
    print(open("/kaggle/working/logs/train_{}.log".format(language), errors="replace").read()[-1500:])

In [ ]:
# Generate all deliverables from the best validation checkpoint
for language, job in jobs.items():
    if job.returncode != 0:
        print("SKIPPED", language)
        continue
    work = "/tmp/tts/" + language
    results = "/kaggle/working/Results/" + language
    ckpt = work + "/ckpt/best.pth.tar"
    print(language, "best step:", __import__("torch").load(ckpt, map_location="cpu", weights_only=False)["step"])
    subprocess.run(["python", "{}/{}/inference.py".format(PROJECT, language), "--work_dir", work,
                    "--ckpt_file", ckpt, "--results_dir", results], check=True)
    subprocess.run(["python", PROJECT + "/scripts/plot_losses.py", "--log_dir", work + "/log",
                    "--out_dir", results, "--name", language], check=True)
    assets = "/kaggle/working/{}/assets".format(language)
    os.makedirs(assets, exist_ok=True)
    for name in ("stats.json", "speakers.json", "target_speaker.txt"):
        shutil.copy(work + "/preprocessed_data/" + name, assets)

if "English" in jobs and jobs["English"].returncode == 0:
    subprocess.run(["python", PROJECT + "/English/inference.py", "--work_dir", "/tmp/tts/English",
                    "--ckpt_file", "/tmp/tts/English/ckpt/best.pth.tar", "--frontend", "stock",
                    "--results_dir", "/kaggle/working/Results/English/stock_frontend_comparison"], check=True)

subprocess.run("pip freeze > /kaggle/working/kaggle_pip_freeze.txt", shell=True)
subprocess.run("du -sh /kaggle/working/*; ls -la /kaggle/working/*/ckpt", shell=True)